# 40 — Inference Test

Evaluate a trained checkpoint on the held-out test split. The canonical path is
`train_segmentation`'s **eval-only** mode, which rebuilds the deterministic block
split, loads the checkpoint, then runs test evaluation + full-scene segmentation
map + per-patch visualisations + a metrics CSV (all logged to MLflow as `eval_*`).

> GPU recommended. Needs a trained checkpoint (`best_model.pth`) under `ml_models/`
> and the processed data locally.

In [ ]:
# Make the pipeline importable as `crop_mapping_pipeline` regardless of the
# checkout directory name (this repo is `cropmap-remote-sensing-exps`; the
# GPU deploy dir is `crop_mapping_pipeline`). Also silence MLflow telemetry.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'crop_mapping_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from crop_mapping_pipeline import config as C
print('Repo :', REPO)
print('Classes:', C.NUM_CLASSES, '| crops:', list(C.CDL_CLASS_NAMES.values()))
print('S2 bands/date:', C.S2_BAND_NAMES)
print('S2 train dir :', C.S2_TRAIN_DIR)
print('CDL train    :', C.CDL_TRAIN)

In [ ]:
import glob
from crop_mapping_pipeline.stages import train_segmentation as T

## 1. Locate a trained checkpoint

In [ ]:
ckpts = sorted(glob.glob(str(C.MODELS_DIR / '**' / 'best_model.pth'), recursive=True))
for c in ckpts: print(c)
assert ckpts, f'No best_model.pth under {C.MODELS_DIR} — train first (notebook 30).'
CKPT = ckpts[0]
print('\nUsing:', CKPT)

## 2. Run eval-only

Set the module-level `EVAL_ONLY_CKPT` (mirrors the `--eval-only` CLI flag), then
call `main()` with the **same** scenario/arch/top-k the checkpoint was trained on —
the split + band selection are rebuilt identically.

In [ ]:
T.EVAL_ONLY_CKPT = CKPT
T.main(
    exps=['gsi'],                       # match the checkpoint's scenario
    archs=['segformer'],                # match the checkpoint's architecture
    top_k=[C.SELECT_TOP_K_PER_CROP],
    skip_ndvi=True,
)
T.EVAL_ONLY_CKPT = None   # reset so later cells train normally

## 3. Outputs

Eval writes into the experiment output dir (and logs to MLflow):
`test_segmentation_map.png`, `test_patches/patch_*.png`, `test_patch_metrics.csv`,
`test_per_class_iou.csv`, `confusion_matrix.png`. Display the segmentation map:

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
maps = sorted(glob.glob(str(C.MODELS_DIR / '**' / 'test_segmentation_map.png'), recursive=True))
maps += sorted(glob.glob(str(C.FIGURES_DIR / '**' / 'test_segmentation_map.png'), recursive=True))
if maps:
    plt.figure(figsize=(14, 6)); plt.imshow(mpimg.imread(maps[-1]))
    plt.axis('off'); plt.title(Path(maps[-1]).parent.name); plt.show()
else:
    print('No segmentation map found yet — run the eval cell above.')

## 4. Advanced — manual tiled inference on arbitrary S2

`run_full_inference()` predicts a full scene tile-by-tile. It needs the trained
`model`, the scenario's `band_indices`, and `band_percentiles` (per-channel 2/98
percentiles computed from training). Reconstructing those outside `main()` is
involved, so prefer eval-only above. Sketch:

```python
model = T.build_model('segformer', in_channels=K, num_classes=C.NUM_CLASSES).to(T.DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=T.DEVICE))
pred_map, profile = T.run_full_inference(
    model, s2_paths, band_indices,
    patch_size=C.PATCH_SIZE, stride=C.PATCH_SIZE,
    band_percentiles=band_percentiles, norm_mode='percentile')
gt_map, _ = T.load_gt_remap(str(C.CDL_TRAIN))
```